# Freeze the 23-class split

Run once. Reruns read the existing manifest. Image-level split with seed 2026 and approximately 70/15/15 per class. Prefer unused validation and test images. If a class has too few, reuse only its corresponding historical validation or test images and record their history. All archive entries are accounted for, including duplicate exclusions and conflicting exact matches. The source label table has no patient or exam ID. Images remain in the official ZIP.

In [ ]:
import csv
import io
from collections import Counter, defaultdict
from hashlib import sha256
from pathlib import Path, PurePosixPath
from random import Random
from shutil import copyfileobj
from zipfile import ZipFile

root = Path("E:/VLM")
archive_path = root / "data/hyper-kvasir-labeled-images.zip"
data_dir = root / "data/hyperkvasir_23_v1"
manifest_path = data_dir / "split_manifest.csv"

if not manifest_path.exists():
    history = defaultdict(set)
    for folder in ("hyperkvasir_lora", "hyperkvasir_lora_expanded", "hyperkvasir_archive_v1"):
        with (root / "data" / folder / "manifest.csv").open(newline="", encoding="utf-8") as handle:
            for row in csv.DictReader(handle):
                member = row.get("archive_member") or row["image"]
                history[PurePosixPath(member).name].add(row["split"])

    with ZipFile(archive_path) as archive:
        members = sorted(PurePosixPath(name) for name in archive.namelist()
                         if name.lower().endswith((".jpg", ".jpeg")))
        metadata = list(csv.DictReader(io.StringIO(
            archive.read("labeled-images/image-labels.csv").decode("utf-8-sig")
        )))
        official = {row["Video file"]: row["Finding"] for row in metadata}
        assert len(members) == len(metadata) == len(official) == 10662
        assert len({path.parent.name for path in members}) == 23
        pilot_rng = Random(42)
        for label in ("z-line", "barretts", "esophagitis-a"):
            cohort = [path for path in members if path.parent.name == label]
            for path in pilot_rng.sample(cohort, 5):
                history[path.name].add("pilot")

        groups = defaultdict(list)
        for member in members:
            label = member.parent.name
            if official[member.stem] != label:
                raise ValueError(f"Label mismatch: {member}")
            digest = sha256(archive.read(member.as_posix())).hexdigest()
            groups[digest].append(member)

    pools = defaultdict(list)
    records = []
    for digest, group in groups.items():
        usages = set().union(*(history[member.name] for member in group))
        prior_use = "|".join(sorted(usages))
        conflict = len({member.parent.name for member in group}) > 1
        canonical = group[0]
        if not conflict:
            pools[canonical.parent.name].append((canonical, digest, usages))
        for member in group if conflict else group[1:]:
            records.append(dict(split="excluded", label=member.parent.name,
                                archive_member=member.as_posix(), sha256=digest,
                                prior_use=prior_use, status="label_conflict" if conflict else "duplicate"))

    rng = Random(2026)
    for label, pool in sorted(pools.items()):
        rng.shuffle(pool)
        fresh = [row for row in pool if not row[2]]
        target = max(1, round(0.15 * len(pool)))
        testing = fresh[:target]
        validation = fresh[target:2 * target]
        used = {row[1] for row in testing + validation}
        for split, selected in (("test", testing), ("validation", validation)):
            for row in pool:
                if len(selected) >= target:
                    break
                if row[1] not in used and row[2] == {split}:
                    selected.append(row)
                    used.add(row[1])
            if not selected:
                raise ValueError(f"No usable {split} image for {label}")
        assignments = {row[1]: "test" for row in testing}
        assignments.update({row[1]: "validation" for row in validation})
        for member, digest, usages in pool:
            split = assignments.get(digest, "train")
            records.append(dict(split=split, label=label, archive_member=member.as_posix(),
                                sha256=digest, prior_use="|".join(sorted(usages)), status="included"))

    included = [row for row in records if row["status"] == "included"]
    assert len(records) == 10662 and len({row["archive_member"] for row in records}) == 10662
    assert len({row["sha256"] for row in included}) == len(included)
    assert all({row["split"] for row in included if row["label"] == label}
               == {"train", "validation", "test"} for label in pools)
    data_dir.mkdir(parents=True, exist_ok=True)
    temporary = manifest_path.with_suffix(".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(sorted(records, key=lambda row: row["archive_member"]))
    temporary.replace(manifest_path)

with manifest_path.open(newline="", encoding="utf-8") as handle:
    records = list(csv.DictReader(handle))
print("Frozen SHA-256:", sha256(manifest_path.read_bytes()).hexdigest())
print("Totals:", dict(Counter(row["split"] for row in records)))
print("Excluded:", dict(Counter(row["status"] for row in records if row["split"] == "excluded")))
print("Previously used evaluation images:", dict(Counter(
    (row["split"], row["label"]) for row in records
    if row["prior_use"] and row["split"] in {"validation", "test"}
)))